This tutorial is meant to be ran in Colab, in a GPU runtime  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://google.com)

In [ ]:
# Clone the repository
!if [ ! -d "ssnr_sim" ]; then git clone -q https://github.com/Balint-H/ssnr_sim.git; fi
%cd /content/ssnr_sim


print("Cloning complete!")


# Training PPO Agents on a Planar Muscle-Tendon Arm Environment using MJX

This notebook adapts the original planar arm model from Day 2 with muscle-tendon actuators to run on the GPU via **MJX** and **Brax**. It loads the model dynamically from your original external XML file and optimizes tracking for an operational workspace reach task.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, please switch via **Runtime > Change runtime type**.


In [ ]:
# @title Extra dependencies for accelerated RL (MuJoCo playground and MJX)
!pip install jax[cuda12] mujoco==3.6 mujoco_mjx==3.6 playground==0.2.0 warp-lang==1.11.0

import distutils.util
import os
import subprocess


try:
  print('Checking that the installation succeeded:')
  import mujoco
  import jax
  print(jax.default_backend())
  !pip list
except Exception as e:
  raise e from RuntimeError(
  'Something went wrong during installation.'
  )

print('Installation successful.')




In [ ]:
# @title Import packages for plotting and creating graphics

import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union, Any, Dict
import numpy as np

print("Installing mediapy")
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:


from datetime import datetime
import functools
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from IPython.display import clear_output
import jax
from jax import numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco_playground._src import mjx_env
from mujoco_playground import wrapper



## Environment Definition

We point directly to your external `arm_model_tendon.xml` file.

The task objective minimizes distance to a targeted operational position via an exponential tracking landscape:

$$R = e^{-\beta \|x_{\text{tip}} - x_{\text{target}}\|^2} - \alpha \|a\|^2$$

---



In [ ]:
# @title Define the MJX learning environment
XML_PATH = './src/ssnr_sim/SSNR2026/arm_model_tendon.xml'

default_config = config_dict.create(
      sim_dt = 0.004,
      ctrl_dt = 0.01,
      episode_length = 400,
      reset_pos_range = [0.8, 0.8],
      reset_vel_range = 0.8,
      reward_falloff_scale = 10,
      reward_fatigue_penalty = 0.05
)

class PlanarArmTendon(mjx_env.MjxEnv):
  """Planar muscle-tendon arm tracking environment running fully on MJX."""

  def __init__(self, config: config_dict.ConfigDict = default_config):
    super().__init__(config)
    self._mj_spec =  mujoco.MjSpec.from_file(filename=XML_PATH, assets=None)
    self._mj_spec = self.preprocess_spec(self._mj_spec)
    self._mj_model = self._mj_spec.compile()

    self._mj_model.opt.timestep = self.sim_dt
    self._mjx_model = mjx.put_model(self._mj_model, impl="warp")
    self._post_init()

  def preprocess_spec(self, spec):
    for a in spec.actuators:
        a.set_to_muscle(lmin=0.5, lmax=1.6, vmax=1.5, fpmax=1.3, fvmax=1.2,
                        timeconst=0.01, tausmooth=0,
                        force=-1, scale=200, range=0.75)
        a.gainprm[1] = 1.05
        a.dynprm[1] = 0.04
        a.biasprm[1] = 1.05
        a.ctrlrange = [0, 1]
    return spec

  def _post_init(self) -> None:
    self._shoulder_qposadr = self._mj_model.joint("shoulder").id
    self._elbow_qposadr = self._mj_model.joint("elbow").id
    self._wrist_qposadr = self._mj_model.joint("wrist").id
    self._tip_body_id = self._mj_model.body("tip").id

  def reset(self, rng: jax.Array) -> mjx_env.State:
    rng, rng_q, rng_v, rng_target1, rng_target2 = jax.random.split(rng, 5)

    low_bounds = self._mj_model.jnt_range[:, 0] * self._config.reset_pos_range[0]
    high_bounds = self._mj_model.jnt_range[:, 1] * self._config.reset_pos_range[1]
    v_bound = ((self._mj_model.jnt_range[:, 1] - self._mj_model.jnt_range[:, 0])
                * self._config.reset_vel_range)
    new_qpos = jax.random.uniform(rng_q, shape=self._mj_model.qpos0.shape,
                                  minval=low_bounds,
                                  maxval=high_bounds)
    new_qvel = jax.random.uniform(rng_v, shape=self._mj_model.qpos0.shape,
                                  minval=-v_bound,
                                  maxval=v_bound)

    data = mjx.make_data(
          self.mj_model,
          impl=self._mjx_model.impl.value, naconmax=3, njmax=6
      )
    data = data.replace(qpos=new_qpos, qvel=new_qvel,)
    data = mjx.forward(self.mjx_model, data)
    data = data.replace(act = jp.ones_like(data.act)*jp.nan)

    radius = jax.random.uniform(rng_target1, (), minval=0.2, maxval=0.5)
    angle = jax.random.uniform(rng_target2, (), minval=-jp.pi/6, maxval=jp.pi/1.5)
    target_pos = jp.array([radius * jp.cos(angle), radius * jp.sin(angle)])
    data = data.replace(mocap_pos = jp.array([[target_pos[0], target_pos[1], 0.]]))
    metrics = {
        "reward/tracking": jp.zeros(()),
        "reward/fatigue_penalty": jp.zeros(()),
        "distance": jp.zeros(())
    }

    info = {"rng": rng, "target": target_pos}
    obs = self._get_obs(data, info)
    reward_val, done = jp.zeros(2)

    return mjx_env.State(data, obs, reward_val, done, metrics, info)


  def step(self, state: mjx_env.State, action: jax.Array) -> mjx_env.State:

    ctrl = jp.clip(action, 0.0, 1.0)
    data = state.data
    data = data.replace(act=jp.where(jp.isnan(data.act), ctrl, data.act))
    data = data.replace(mocap_pos = jp.array([[data.mocap_pos[0][0], data.mocap_pos[0][1], 0.]]))

    data = mjx_env.step(self.mjx_model, data, ctrl, self.n_substeps)

    reward_val, metrics = self._get_reward(data, ctrl, state.info, state.metrics)
    obs = self._get_obs(data, state.info)

    done = self._get_done(data)

    return state.replace(data=data, obs=obs, reward=reward_val, done=done, metrics=metrics)

  def _get_done(self, data):
    done = jp.isnan(data.qpos).any() | jp.isnan(data.qvel).any()
    at_low = data.qpos <= self._mjx_model.jnt_range[:, 0]
    at_high = data.qpos >= self._mjx_model.jnt_range[:, 1]
    done = done | jp.any(at_low | at_high)
    done = done.astype(float)
    return done

  def _get_obs(self, data: mjx.Data, info: dict[str, Any]) -> jax.Array:
    tip_pos = data.xpos[self._tip_body_id][:2]
    target_pos = data.mocap_pos[0][:2]


    return jp.concatenate([
        jp.sin(data.qpos),
        jp.cos(data.qpos),
        data.qvel,
        tip_pos,
        target_pos,
        target_pos - tip_pos,
    ])



  def _get_reward(self, data: mjx.Data, action: jax.Array, info: dict[str, Any], metrics: dict[str, Any]) -> jax.Array:
    tip_pos = data.xpos[self._tip_body_id][:2]
    target_pos = data.mocap_pos[0][:2]

    dist = jp.linalg.norm(tip_pos - target_pos)
    metrics["distance"] = dist

    tracking = jp.exp(-self._config.reward_falloff_scale * dist)
    metrics["reward/tracking"] = tracking

    fatigue_penalty = -(self._config.reward_fatigue_penalty
                           * jp.sum(jp.square(action)))
    metrics["reward/fatigue_penalty"] = fatigue_penalty

    return tracking + fatigue_penalty, metrics


  @property
  def xml_path(self) -> str:
    return XML_PATH

  @property
  def action_size(self) -> int:
    return self.mjx_model.nu

  @property
  def mj_model(self) -> mujoco.MjModel:
    return self._mj_model

  @property
  def mjx_model(self) -> mjx.Model:
    return self._mjx_model

## JAX Unroll Diagnostic Initialization Test

Before running the full policy training sequence, we double-check state processing directly using JAX compilation transformations.

---


In [ ]:
env = PlanarArmTendon()
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

print("Compile and reset")
state = jit_reset(jax.random.PRNGKey(14))
print("Compile and step")
state = jit_step(state, jp.zeros(env.action_size))
print("Environment compiled successfully")
print("Observations Vector Dimensions:", state.obs.shape)
print("Observation Allocation Engine target device:", state.obs.device)


In [ ]:
v_jit_reset = jax.jit(jax.vmap(env.reset))
v_jit_step = jax.jit(jax.vmap(env.step))
v_keys = jax.random.split(jax.random.PRNGKey(14),10)
v_keys

In [ ]:
v_states = v_jit_reset(v_keys)
v_states = v_jit_step(v_states, jp.zeros((10, env.action_size)))
v_states.data.qpos.shape


## Training the Agent with PPO

We parse the environment definitions down to the optimized `brax.training` processing stack.

---



In [ ]:

ppo_params = {
"num_timesteps": 20_000_000,
"num_envs": 4096,
"unroll_length": 10,
"discounting": 0.97,
"learning_rate": 2.5e-4,
"batch_size": 256,
"num_minibatches":32,
"entropy_cost": 1e-3,
"num_evals": 16,
"seed": 0,
"episode_length": default_config.episode_length,
}

network_params = {
    "policy_hidden_layer_sizes": (128, 64),
    "value_hidden_layer_sizes": (128, 64),
}

x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]

def progress(num_steps, metrics):
  print(num_steps)
  clear_output(wait=True)
  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.figure(figsize=(10, 5))
  plt.xlim([0, ppo_params["num_timesteps"] * 1.1])
  plt.xlabel("# Environment Steps")
  plt.ylabel("Reward per Episode")
  plt.title(f"Step: {num_steps} -> Mean Evaluation Reward: {y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="orange")
  display(plt.gcf())
  plt.close()

network_factory = functools.partial(ppo_networks.make_ppo_networks, **network_params)

train_fn = functools.partial(
ppo.train,
**ppo_params,
network_factory=network_factory,
progress_fn=progress,
wrap_env_fn=wrapper.wrap_for_brax_training,
normalize_observations=True
)

from mujoco_playground import wrapper
print("Running training...")
make_inference_fn, params, metrics = train_fn(
environment=env,
wrap_env_fn=wrapper.wrap_for_brax_training,
)

print(f"Time to compile (JIT): {times[1] - times[0]}")
print(f"Time to complete training: {times[-1] - times[1]}")

import pickle
with open('playground_params.pickle', 'wb') as handle:
  pickle.dump(params, handle, protocol=pickle.HIGHEST_PROTOCOL)